# Basic Usage of AMGD for Poisson Regression

This notebook demonstrates how to use the Adaptive Momentum Gradient Descent (AMGD) algorithm for regularized Poisson regression.

## Overview

AMGD combines adaptive learning rates, momentum, and adaptive soft-thresholding to address key limitations of existing methods like AdaGrad and Adam when applied to regularized GLMs, particularly Poisson regression.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import sys
import os

# Add the parent directory to path to enable imports
sys.path.append(os.path.join(os.getcwd(), '..'))

# Import our AMGD implementation
from scripts.amgd_implementation import amgd, evaluate_model, poisson_log_likelihood
from scripts.data_processing import preprocess_ecological_dataset

# Set plotting style
plt.style.use('ggplot')
sns.set_palette('colorblind')

## 1. Load and Preprocess the Data

We'll use the Ecological Health Dataset, focusing on predicting the Biodiversity_Index.

In [ ]:
# Load and preprocess data
try:
    X, y, feature_names = preprocess_ecological_dataset("../data/ecological_health_dataset.csv")
    print(f"Data loaded successfully: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"Feature names: {feature_names}")
except FileNotFoundError:
    print("Dataset file not found. Please download it from Kaggle and place it in the data directory.")
    print("Link: https://www.kaggle.com/datasets/datasetengineer/ecological-health-dataset")

## 2. Split the Data

We'll split the data into training and testing sets (80/20).

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

## 3. Train the AMGD Model

Now we'll train a Poisson regression model using the AMGD algorithm with L1 regularization.

In [ ]:
# Set parameters for AMGD
params = {
    "alpha": 0.01,          # Learning rate
    "beta1": 0.9,           # First moment decay rate
    "beta2": 0.999,         # Second moment decay rate
    "lambda1": 0.1,         # L1 regularization strength
    "lambda2": 0.0,         # L2 regularization strength (for ElasticNet)
    "penalty": "l1",        # 'l1' or 'elasticnet'
    "T": 100.0,             # Gradient clipping threshold
    "tol": 1e-6,            # Convergence tolerance
    "max_iter": 500,        # Maximum iterations
    "eta": 0.0001,          # Learning rate decay
    "epsilon": 1e-8,        # Numerical stability constant
    "verbose": True,        # Print progress
    "return_iters": True    # Return beta history
}

# Train the model using AMGD
print("Training AMGD model...")
beta, loss_history, runtime, nonzero_history, beta_history = amgd(X_train, y_train, **params)

print(f"Training completed in {runtime:.4f} seconds")
print(f"Final loss: {loss_history[-1]:.4f}")
print(f"Non-zero coefficients: {np.sum(np.abs(beta) > 1e-6)} out of {len(beta)}")
print(f"Sparsity: {1 - np.sum(np.abs(beta) > 1e-6) / len(beta):.4f}")

## 4. Evaluate the Model

Now let's evaluate the model's performance on the test set.

In [ ]:
# Evaluate the model on test data
test_metrics = evaluate_model(beta, X_test, y_test)

print("\nTest metrics:")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

## 5. Visualize Training Convergence

Let's visualize how the loss decreases during training.

In [ ]:
# Plot loss convergence
plt.figure(figsize=(10, 6))
plt.plot(loss_history, color='#3498db', linewidth=2)
plt.xlabel('Iterations', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('AMGD Training Convergence', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.yscale('log')  # Use log scale for better visualization
plt.tight_layout()
plt.show()

# Plot sparsity evolution
plt.figure(figsize=(10, 6))
plt.plot(nonzero_history, color='#e74c3c', linewidth=2)
plt.xlabel('Iterations', fontsize=12)
plt.ylabel('Number of Non-Zero Coefficients', fontsize=12)
plt.title('Feature Selection During Training', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 6. Visualize Feature Importance

Let's visualize the importance of each feature based on the coefficient magnitudes.

In [ ]:
# Calculate feature importance
importance = np.abs(beta)
indices = np.argsort(importance)[::-1]

# Get the top features
top_n = 15
top_indices = indices[:top_n]
top_importances = importance[top_indices]
top_feature_names = [feature_names[i] if i < len(feature_names) else f"Feature {i}" for i in top_indices]

# Plot feature importance
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_indices)), top_importances, align='center', color='#3498db')
plt.yticks(range(len(top_indices)), top_feature_names)
plt.xlabel('Coefficient Magnitude', fontsize=12)
plt.title('Top Feature Importances (AMGD with L1 Regularization)', fontsize=14)
plt.gca().invert_yaxis()  # Highest importance at the top
plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we demonstrated how to use the AMGD algorithm for Poisson regression with L1 regularization. We showed how to:

1. Load and preprocess the Ecological Health Dataset
2. Train a Poisson regression model using AMGD
3. Evaluate the model's performance on test data
4. Visualize training convergence and feature importance

AMGD effectively performs feature selection while achieving good predictive performance, as evidenced by the sparsity of the final model and the test metrics.